# SARBLOH: DeepSeek-V4-Flash server probe (no games)

Starts `model: "deepseek_v4"` (`Sarbloh-Experimentation/harness/llm/deepseek_v4.py`) on the RTX PRO 6000 and measures
what decides whether it can replace Qwen3.8-Flash-Next:
1. Does SGLang start it, and how long does the start take (the bar: 30 minutes)?
2. Decode tok/s with 1 and 10 requests at once (Flash-Next on the same card: about 156 and 780).
3. The prefix-cache hit rate in the agent's own shape: the ipython tool on every request, thinking on and sent back.
4. The full DeepSeek-V4-Flash-Vision-Exp (168 GB) with CPU offload, served text only: start time and tok/s.

The `harness/llm` source is embedded in the source cell by `scripts/kaggle_push.py dsv4-probe` (no agent dataset
needed). Internet off. Outputs: `dsv4/server_bench.json`, `dsv4/server_events.jsonl`, `dsv4/llm-server.log`.

## 0. CONFIG

In [ ]:
import time
NOTEBOOK_START = time.time()

BUDGET_S = 4.0 * 3600            # hard stop for the whole probe
STREAMS = 10                     # the agent runs 10 games at once
START_TARGET_S = 30 * 60         # the server must be up within 30 minutes
# REAP (fits the GPU): the first profile that starts gets the full bench
REAP_CHAIN = ["dsv4_reap150", "dsv4_reap150_marlin", "dsv4_reap150_pennyroyal"]
# the full Vision-Exp with its weights partly in CPU RAM: a short bench, if at least OFFLOAD_MIN_S are left
# Empty in v2: v1 (REAP + the 168 GB Vision-Exp attached) failed before any cell ran; Vision-Exp gets its own run.
OFFLOAD_CHAIN = []  # ["dsv4_vision_exp_offload_dspark", "dsv4_vision_exp_offload"], with VISION_EXP attached
OFFLOAD_MIN_S = 90 * 60
SERVER = {"port": 8000, "startup_timeout_s": 3600.0, "max_inflight": 32, "watchdog_cfg": {}}
print("probe started", time.ctime(), flush=True)

## 1. Source (embedded at push time) and environment

In [ ]:
# @@PRIME_LLM_SOURCES@@
import subprocess, sys
from pathlib import Path

SRC = Path("/tmp/sarbloh-src")
for rel, text in SOURCES.items():
    p = SRC / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding="utf-8")
sys.path.insert(0, str(SRC))
print("source:", SOURCES_META)
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
_vram_mib = int(subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
                               capture_output=True, text=True).stdout.split()[0])
assert _vram_mib > 90_000, f"wrong GPU: {_vram_mib} MiB; the probe needs the RTX Pro 6000 (machine_shape NvidiaRtxPro6000)"
for cmd in ("free -g", "nproc", "lscpu | grep -E 'Model name|Socket|NUMA|Flags' | cut -c1-200",
            "df -h /tmp /kaggle/working", "ls -la /kaggle/input/models/*/* 2>/dev/null | head -40"):
    print("$", cmd)
    print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout)

## 2. REAP-150B: start + full bench

In [ ]:
import json, traceback
from harness.llm import bench
from harness.llm.server import LlmServer
from harness.llm.spec import get_spec

OUT = Path("/kaggle/working/dsv4")
SPEC = get_spec("deepseek_v4")
DEADLINE = NOTEBOOK_START + BUDGET_S
ROWS = []
for profile in REAP_CHAIN:
    print(f"==== {profile} at {time.time() - NOTEBOOK_START:.0f} s", flush=True)
    try:
        ROWS = bench.run("deepseek_v4", [profile], SERVER, {}, OUT, DEADLINE, streams=STREAMS, stress_minutes=0)
    except Exception:
        traceback.print_exc()
    if ROWS and ROWS[-1].get("profile") == profile and "error" not in ROWS[-1] and "skipped" not in ROWS[-1]:
        break

## 3. Vision-Exp with CPU offload: start + short bench

In [ ]:
def short_bench(profile):
    server = LlmServer(SPEC, SERVER, OUT, deadline=DEADLINE)
    row = {"model": "deepseek_v4", "profile": profile}
    try:
        server.start(chain=[profile])
        row.update(startup_s=server.startup_s, tool_mode=server.tool_mode)
        for name, fn in (("decode_1", lambda: bench.decode(server, 1, max_tokens=128)),
                         (f"decode_{STREAMS}", lambda: bench.decode(server, STREAMS, max_tokens=128)),
                         ("agent_like_tools", lambda: bench.agent_like(server, STREAMS, turns=2, max_tokens=512,
                                                                       tools=True))):
            if time.time() > DEADLINE - 600:
                row[name] = {"skipped": "time"}
                continue
            try:
                row[name] = fn()
            except Exception as exc:
                row[name] = {"error": repr(exc)[:500]}
            print(profile, name, json.dumps(row[name], default=str)[:400], flush=True)
    except Exception as exc:
        row.update(error=repr(exc)[:800], attempts=server.attempts)
    finally:
        server.stop()
    return row

for profile in OFFLOAD_CHAIN:
    left = DEADLINE - time.time()
    if left < OFFLOAD_MIN_S:
        ROWS.append({"model": "deepseek_v4", "profile": profile, "skipped": f"time ({left / 60:.0f} min left)"})
        break
    print(f"==== {profile} at {time.time() - NOTEBOOK_START:.0f} s", flush=True)
    ROWS.append(short_bench(profile))
    (OUT / "server_bench.json").write_text(json.dumps(ROWS, indent=1, default=str))
    if "error" not in ROWS[-1]:
        break

## 4. Summary

In [ ]:
import re
bench.print_table(ROWS, STREAMS)
print()
for r in ROWS:
    s = r.get("startup_s")
    verdict = "no start" if s is None else ("OK" if s <= START_TARGET_S else "SLOW")
    print(f"start {r.get('profile')}: {s} s ({verdict}, target {START_TARGET_S} s)  {str(r.get('error', ''))[:200]}")
ev = OUT / "server_events.jsonl"
if ev.exists():
    for line in ev.read_text().splitlines():
        e = json.loads(line)
        if e["event"] in ("runtime_ready", "launch", "ready", "start_failed"):
            print(json.dumps(e, default=str)[:1500])

# The server log, one block per profile: memory, weight load and graph capture lines, then the prefix-cache hit
# rate over every prefill batch and the decode throughput by number of running requests (as read for Flash-Next).
log = (OUT / "llm-server.log").read_text(errors="replace") if (OUT / "llm-server.log").exists() else ""
KEYS = ("max_total_num_tokens", "Load weight end", "avail mem", "KV Cache", "Capture cuda graph end", "moe_runner",
        "flashinfer_mxfp4", "page_size", "swa_full_tokens_ratio", "is_fp4_experts", "cpu_offload", "Traceback",
        "Error", "error:")
PREFILL = re.compile(r"Prefill batch.*?#new-token: (\d+), #cached-token: (\d+)")
DECODE = re.compile(r"Decode batch, #running-req: (\d+).*?gen throughput \(token/s\): ([0-9.]+)")
LOGSTATS = {}
for block in log.split("===== profile=")[1:]:
    name = block.split()[0]
    print(f"\n==== log {name}")
    for line in block.splitlines():
        if any(k in line for k in KEYS):
            print("  " + line[:300])
    new = cached = 0
    for a, b in PREFILL.findall(block):
        new, cached = new + int(a), cached + int(b)
    by_run = {}
    for k, v in DECODE.findall(block):
        by_run.setdefault(int(k), []).append(float(v))
    LOGSTATS[name] = {"prefill_cache_hit": round(cached / max(1, new + cached), 3), "prefill_tokens": new + cached,
                      "decode_tok_s_by_running": {k: round(sum(v) / len(v), 1) for k, v in sorted(by_run.items())}}
    print("  ", json.dumps(LOGSTATS[name]))
print("DSV4_RESULT " + json.dumps({"rows": ROWS, "log": LOGSTATS}, default=str)[:30000])
print(f"notebook wall: {time.time() - NOTEBOOK_START:.0f}s")